# BBC Articles Two-Stage Classification: Sport / Non-Sport -> Individual Sport

Hierarchical classification pipeline with compact outputs and additional exploratory plots

how it works:

- Read ds
- Stage 1: classify each article as `sport` or `non-sport`
- Stage 2: only for articles predicted as `sport`, classify the specific sport
- unify title + text = combined_text
- use train/validation/test splits for both stages

TEXT VECTORIZATION

it doesnt use the text as string BUT it converts it in numbers.

- lowercase, without accents, REMOVE 'the/and/is...'
- ngram_range=(1,2) -> single and couple worlds 'premier league, grand slam, rugby world cup...'
- makes common worls across every article weight less ('the/is...')
- makes topic-specific worlds weight MORE ('goal', 'wicket', 'serve', 'scrum', 'stock market'...)

PIPELINE IDEA:

- first model decides if the article is about sport at all
- second model specializes only on sport articles and predicts the single sport
- if stage 1 says `non-sport`, the pipeline stops there


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !pip install -q pandas scikit-learn matplotlib seaborn

import warnings
import re
from collections import Counter
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11


## 2. Load and view ds

In [ ]:
DATA_PATH = '/content/drive/MyDrive/Dataset_Colab/bbc_articles_new_urls.csv'

df = pd.read_csv(DATA_PATH, sep=';')

print(f'Dataset shape: {df.shape}')
display(df.head(3))
print('\nOriginal category distribution:')
print(df['category'].value_counts())


## 3. Targets and text preparation

In [ ]:
def extract_sport_label(url):
    if pd.isna(url) or not isinstance(url, str):
        return np.nan

    if not url.strip():
        return np.nan

    parts = [part for part in urlparse(url).path.split('/') if part]
    if len(parts) >= 2 and parts[0] == 'sport':
        if parts[1] == 'articles':
            return np.nan
        return parts[1]
    return np.nan

# Remap all categories that are not 'sport' to 'non-sport'
df['category'] = df['category'].apply(lambda x: 'sport' if x == 'sport' else 'non-sport')

df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
df['combined_text'] = (df['title'] + ' ' + df['text']).str.strip()

df['sport_label'] = np.where(
    df['category'] == 'sport',
    df['url'].apply(extract_sport_label),
    'non-sport'
)

df = df[df['sport_label'].notna()].copy()

df['binary_label'] = df['category'].map({'non-sport': 0, 'sport': 1})
df['final_label'] = np.where(df['category'] == 'sport', df['sport_label'], 'non-sport')

df['title_len'] = df['title'].str.split().str.len()
df['text_len'] = df['text'].str.split().str.len()
df['combined_len'] = df['combined_text'].str.split().str.len()
df['char_len'] = df['combined_text'].str.len()

sport_class_counts = df[df['category'] == 'sport']['sport_label'].value_counts().sort_values(ascending=False)

summary = pd.DataFrame({
    'metric': ['Rows kept', 'Empty combined_text rows', 'Binary classes', 'Sport classes'],
    'value': [len(df), int((df['combined_text'].str.len() == 0).sum()), df['category'].nunique(), sport_class_counts.size]
})
display(summary)


### 3.1 Data Exploration

Grafici utili per capire bilanciamento, lunghezza degli articoli e parole ricorrenti prima del training.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
order = df['category'].value_counts().index
sns.countplot(x='category', data=df, order=order, ax=ax)
ax.set_title('Binary Class Distribution')
ax.set_xlabel('Category')
ax.set_ylabel('Articles')
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

sport_counts = sport_class_counts.reset_index()
sport_counts.columns = ['sport_label', 'count']
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=sport_counts, x='sport_label', y='count', ax=ax)
ax.set_title('Individual Sport Class Distribution')
ax.set_xlabel('Sport label')
ax.set_ylabel('Articles')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

length_summary = df.groupby('category')[['title_len', 'text_len', 'combined_len']].agg(['mean', 'median', 'min', 'max']).round(1)
display(length_summary)

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=df, x='combined_len', hue='category', bins=40, kde=True, element='step', ax=ax)
ax.set_title('Article Length Distribution by Binary Class')
ax.set_xlabel('Words in title + text')
ax.set_ylabel('Articles')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df[df['category'] == 'sport'], x='sport_label', y='combined_len', ax=ax)
ax.set_title('Article Length by Sport Class')
ax.set_xlabel('Sport label')
ax.set_ylabel('Words in title + text')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

def top_words(texts, n=20):
    tokens = []
    for text in texts.dropna().astype(str):
        tokens.extend([
            token for token in re.findall(r'[a-z]{3,}', text.lower())
            if token not in ENGLISH_STOP_WORDS
        ])
    return pd.DataFrame(Counter(tokens).most_common(n), columns=['word', 'count'])

top_sport_words = top_words(df.loc[df['category'] == 'sport', 'combined_text'])
top_non_sport_words = top_words(df.loc[df['category'] == 'non-sport', 'combined_text'])

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=top_sport_words, y='word', x='count', ax=ax)
ax.set_title('Top Words in Sport Articles')
ax.set_xlabel('Count')
ax.set_ylabel('Word')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=top_non_sport_words, y='word', x='count', ax=ax)
ax.set_title('Top Words in Non-Sport Articles')
ax.set_xlabel('Count')
ax.set_ylabel('Word')
plt.tight_layout()
plt.show()


## 4. Train / Validation / Test Split

In [ ]:
def split_single_class(group, random_state=RANDOM_STATE):
    shuffled = group.sample(frac=1, random_state=random_state)
    n = len(shuffled)

    if n == 1:
        train_n, val_n, test_n = 1, 0, 0
    elif n == 2:
        train_n, val_n, test_n = 1, 0, 1
    elif n == 3:
        train_n, val_n, test_n = 1, 1, 1
    elif n == 4:
        train_n, val_n, test_n = 2, 1, 1
    else:
        val_n = max(1, int(round(n * 0.20)))
        test_n = max(1, int(round(n * 0.20)))
        train_n = n - val_n - test_n

        if train_n < 1:
            train_n = 1
            if val_n >= test_n and val_n > 1:
                val_n -= 1
            else:
                test_n -= 1

    train_split = shuffled.iloc[:train_n]
    val_split = shuffled.iloc[train_n:train_n + val_n]
    test_split = shuffled.iloc[train_n + val_n:train_n + val_n + test_n]

    return train_split, val_split, test_split


# split non-sport with a standard stratified split
non_sport_df = df[df['category'] == 'non-sport'].copy()
ns_train_val, ns_test = train_test_split(
    non_sport_df,
    test_size=0.20,
    random_state=RANDOM_STATE
)
ns_train, ns_val = train_test_split(
    ns_train_val,
    test_size=0.25,
    random_state=RANDOM_STATE
)

# split sport per single sport so every discovered sport stays in training
sport_df = df[df['category'] == 'sport'].copy()

# Filter out less significant sports
min_samples_per_sport = 10
sport_counts = sport_df['sport_label'].value_counts()
significant_sports = sport_counts[sport_counts >= min_samples_per_sport].index.tolist()
sport_df = sport_df[sport_df['sport_label'].isin(significant_sports)].copy()

print(f'\nSport distribution after filtering (min samples = {min_samples_per_sport}):')
print(sport_df['sport_label'].value_counts())

sport_train_parts = []
sport_val_parts = []
sport_test_parts = []

for _, group in sport_df.groupby('sport_label', sort=False):
    train_part, val_part, test_part = split_single_class(group)
    sport_train_parts.append(train_part)
    if len(val_part) > 0:
        sport_val_parts.append(val_part)
    if len(test_part) > 0:
        sport_test_parts.append(test_part)

sport_train_df = pd.concat(sport_train_parts)
sport_val_df = pd.concat(sport_val_parts)
sport_test_df = pd.concat(sport_test_parts)

# final splits used by stage 1 (binary)
train_df = pd.concat([ns_train, sport_train_df]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
val_df = pd.concat([ns_val, sport_val_df]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_df = pd.concat([ns_test, sport_test_df]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Train size: {len(train_df)}')
print(f'Validation size: {len(val_df)}')
print(f'Test size: {len(test_df)}')

print('\nTrain binary balance:')
print(train_df['category'].value_counts())
print('\nValidation binary balance:')
print(val_df['category'].value_counts())
print('\nTest binary balance:')
print(test_df['category'].value_counts())

print('\nTrain sport balance:')
print(sport_train_df['sport_label'].value_counts())
print('\nValidation sport balance:')
print(sport_val_df['sport_label'].value_counts())
print('\nTest sport balance:')
print(sport_test_df['sport_label'].value_counts())

In [ ]:
# Visual check: class balance across train/validation/test
split_overview = pd.concat([
    train_df.assign(split='train'),
    val_df.assign(split='validation'),
    test_df.assign(split='test')
])

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=split_overview, x='split', hue='category', order=['train', 'validation', 'test'], ax=ax)
ax.set_title('Binary Class Balance by Split')
ax.set_xlabel('Split')
ax.set_ylabel('Articles')
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

sport_split_overview = pd.concat([
    sport_train_df.assign(split='train'),
    sport_val_df.assign(split='validation'),
    sport_test_df.assign(split='test')
])

fig, ax = plt.subplots(figsize=(12, 5))
sns.countplot(data=sport_split_overview, x='sport_label', hue='split', ax=ax)
ax.set_title('Sport Class Balance by Split')
ax.set_xlabel('Sport label')
ax.set_ylabel('Articles')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## 5. Stage 1: Sport vs Non-Sport

In [ ]:
X_train_bin = train_df['combined_text']
y_train_bin = train_df['binary_label']

X_val_bin = val_df['combined_text']
y_val_bin = val_df['binary_label']

X_test_bin = test_df['combined_text']
y_test_bin = test_df['binary_label']

### TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

binary_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words='english',
    ngram_range=(1, 2),
    max_features=30000,
    min_df=2
)

X_train_bin_vec = binary_vectorizer.fit_transform(X_train_bin)
X_val_bin_vec   = binary_vectorizer.transform(X_val_bin)
X_test_bin_vec  = binary_vectorizer.transform(X_test_bin)

VECTORIZER_TYPE = 'TF-IDF'
print(f'[{VECTORIZER_TYPE}] Train shape: {X_train_bin_vec.shape}')
print(f'[{VECTORIZER_TYPE}] Val   shape: {X_val_bin_vec.shape}')
print(f'[{VECTORIZER_TYPE}] Test  shape: {X_test_bin_vec.shape}')

In [ ]:
# Most informative TF-IDF terms in the binary training matrix
mean_tfidf = np.asarray(X_train_bin_vec.mean(axis=0)).ravel()
feature_names = np.array(binary_vectorizer.get_feature_names_out())
top_idx = mean_tfidf.argsort()[-20:][::-1]
top_tfidf_terms = pd.DataFrame({'term': feature_names[top_idx], 'mean_tfidf': mean_tfidf[top_idx]})

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=top_tfidf_terms, y='term', x='mean_tfidf', ax=ax)
ax.set_title('Top Mean TF-IDF Terms - Stage 1 Training Set')
ax.set_xlabel('Mean TF-IDF')
ax.set_ylabel('Term')
plt.tight_layout()
plt.show()


In [ ]:
!pip install -q --upgrade gensim


### Word2Vec

In [ ]:
import re
import numpy as np
import gensim.downloader as gensim_api

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

_stop_en = set(stopwords.words('english'))

def _tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return [t for t in text.split() if t not in _stop_en and len(t) > 1]

def _docs_to_matrix_pretrained(texts, model, vector_size: int) -> np.ndarray:
    rows = []
    for text in texts:
        tokens = _tokenize(text)
        vecs = [model[t] for t in tokens if t in model]
        rows.append(np.mean(vecs, axis=0) if vecs else np.zeros(vector_size))
    return np.array(rows)

print('Loading fastText model...')
ft_model = gensim_api.load('fasttext-wiki-news-subwords-300')
W2V_VECTOR_SIZE = 300

X_train_bin_vec = _docs_to_matrix_pretrained(X_train_bin, ft_model, W2V_VECTOR_SIZE)
X_val_bin_vec   = _docs_to_matrix_pretrained(X_val_bin,   ft_model, W2V_VECTOR_SIZE)
X_test_bin_vec  = _docs_to_matrix_pretrained(X_test_bin,  ft_model, W2V_VECTOR_SIZE)

VECTORIZER_TYPE = 'fastText pretrained (300d)'
print(f'[{VECTORIZER_TYPE}] Train shape: {X_train_bin_vec.shape}')
print(f'[{VECTORIZER_TYPE}] Val   shape: {X_val_bin_vec.shape}')
print(f'[{VECTORIZER_TYPE}] Test  shape: {X_test_bin_vec.shape}')

In [ ]:
binary_mlp = MLPClassifier(
    hidden_layer_sizes=(256, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    learning_rate_init=1e-3,
    max_iter=1,
    warm_start=True,
    random_state=RANDOM_STATE
)

EPOCHS = 20
binary_train_acc_history = []
binary_val_acc_history = []

for epoch in range(EPOCHS):
    binary_mlp.fit(X_train_bin_vec, y_train_bin)

    train_pred = binary_mlp.predict(X_train_bin_vec)
    val_pred = binary_mlp.predict(X_val_bin_vec)

    binary_train_acc_history.append(accuracy_score(y_train_bin, train_pred))
    binary_val_acc_history.append(accuracy_score(y_val_bin, val_pred))

binary_history = pd.DataFrame({
    'epoch': np.arange(1, EPOCHS + 1),
    'train_accuracy': binary_train_acc_history,
    'validation_accuracy': binary_val_acc_history
})
display(binary_history.tail())


In [ ]:
best_binary_epoch = int(np.argmax(binary_val_acc_history))
best_binary_val_acc = binary_val_acc_history[best_binary_epoch]

print(f'Best stage 1 validation accuracy: {best_binary_val_acc:.4f} at epoch {best_binary_epoch + 1}')

plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), binary_train_acc_history, label='Train Accuracy')
plt.plot(range(1, EPOCHS + 1), binary_val_acc_history, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Stage 1: Training vs Validation Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
y_test_bin_pred = binary_mlp.predict(X_test_bin_vec)

test_acc = accuracy_score(y_test_bin, y_test_bin_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test_bin,
    y_test_bin_pred,
    average='binary',
    zero_division=0
)

print(f'Stage 1 Test Accuracy : {test_acc:.4f}')
print(f'Stage 1 Test Precision: {precision:.4f}')
print(f'Stage 1 Test Recall   : {recall:.4f}')
print(f'Stage 1 Test F1-score : {f1:.4f}')

print("=============================================================")
print(f'[{VECTORIZER_TYPE}] Stage 1')
print('\nDetailed classification report:')
print()
print(classification_report(y_test_bin, y_test_bin_pred, target_names=['Non-Sport', 'Sport'], zero_division=0))

cm = confusion_matrix(y_test_bin, y_test_bin_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Non-Sport', 'Sport'],
    yticklabels=['Non-Sport', 'Sport']
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Stage 1 Confusion Matrix (Test Set)')
plt.show()


## 6. Stage 2: Individual Sport Classification

In [ ]:
sport_class_names = sorted(sport_df['sport_label'].unique())
sport_label_to_id = {label: idx for idx, label in enumerate(sport_class_names)}
sport_id_to_label = {idx: label for label, idx in sport_label_to_id.items()}

sport_train_df = sport_train_df.copy()
sport_val_df = sport_val_df.copy()
sport_test_df = sport_test_df.copy()

sport_train_df['sport_id'] = sport_train_df['sport_label'].map(sport_label_to_id)
sport_val_df['sport_id'] = sport_val_df['sport_label'].map(sport_label_to_id)
sport_test_df['sport_id'] = sport_test_df['sport_label'].map(sport_label_to_id)

X_train_sport = sport_train_df['combined_text']
y_train_sport = sport_train_df['sport_id']

X_val_sport = sport_val_df['combined_text']
y_val_sport = sport_val_df['sport_id']

X_test_sport = sport_test_df['combined_text']
y_test_sport = sport_test_df['sport_id']



In [ ]:
sport_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words='english',
    ngram_range=(1, 2),
    max_features=30000,
    min_df=2
)

X_train_sport_vec = sport_vectorizer.fit_transform(X_train_sport)
X_val_sport_vec   = sport_vectorizer.transform(X_val_sport)
X_test_sport_vec  = sport_vectorizer.transform(X_test_sport)

SPORT_VECTORIZER_TYPE = 'TF-IDF'
print(f'[{SPORT_VECTORIZER_TYPE}] Train shape: {X_train_sport_vec.shape}')
print(f'[{SPORT_VECTORIZER_TYPE}] Val   shape: {X_val_sport_vec.shape}')
print(f'[{SPORT_VECTORIZER_TYPE}] Test  shape: {X_test_sport_vec.shape}')

In [ ]:
X_train_sport_vec = _docs_to_matrix_pretrained(X_train_sport, ft_model, W2V_VECTOR_SIZE)
X_val_sport_vec = _docs_to_matrix_pretrained(X_val_sport,   ft_model, W2V_VECTOR_SIZE)
X_test_sport_vec = _docs_to_matrix_pretrained(X_test_sport,  ft_model, W2V_VECTOR_SIZE)

SPORT_VECTORIZER_TYPE = 'fastText pretrained (300d)'
print(f'[{SPORT_VECTORIZER_TYPE}] Train shape: {X_train_sport_vec.shape}')
print(f'[{SPORT_VECTORIZER_TYPE}] Val   shape: {X_val_sport_vec.shape}')
print(f'[{SPORT_VECTORIZER_TYPE}] Test  shape: {X_test_sport_vec.shape}')

In [ ]:
sport_mlp = MLPClassifier(
    hidden_layer_sizes=(256, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    learning_rate_init=1e-3,
    max_iter=1,
    warm_start=True,
    random_state=RANDOM_STATE
)

sport_train_acc_history = []
sport_val_acc_history = []

for epoch in range(EPOCHS):
    sport_mlp.fit(X_train_sport_vec, y_train_sport)

    train_pred = sport_mlp.predict(X_train_sport_vec)
    val_pred = sport_mlp.predict(X_val_sport_vec)

    sport_train_acc_history.append(accuracy_score(y_train_sport, train_pred))
    sport_val_acc_history.append(accuracy_score(y_val_sport, val_pred))

sport_history = pd.DataFrame({
    'epoch': np.arange(1, EPOCHS + 1),
    'train_accuracy': sport_train_acc_history,
    'validation_accuracy': sport_val_acc_history
})
display(sport_history.tail())


In [ ]:
best_sport_epoch = int(np.argmax(sport_val_acc_history))
best_sport_val_acc = sport_val_acc_history[best_sport_epoch]

print(f'Best stage 2 validation accuracy: {best_sport_val_acc:.4f} at epoch {best_sport_epoch + 1}')

plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), sport_train_acc_history, label='Train Accuracy')
plt.plot(range(1, EPOCHS + 1), sport_val_acc_history, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Stage 2: Training vs Validation Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
y_test_sport_pred = sport_mlp.predict(X_test_sport_vec)

test_acc = accuracy_score(y_test_sport, y_test_sport_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test_sport,
    y_test_sport_pred,
    average='macro',
    zero_division=0
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_test_sport,
    y_test_sport_pred,
    average='weighted',
    zero_division=0
)

print(f'Stage 2 Test Accuracy        : {test_acc:.4f}')
print(f'Stage 2 Test Precision Macro : {precision_macro:.4f}')
print(f'Stage 2 Test Recall Macro    : {recall_macro:.4f}')
print(f'Stage 2 Test F1-score Macro  : {f1_macro:.4f}')
print(f'Stage 2 Test Precision Weight: {precision_weighted:.4f}')
print(f'Stage 2 Test Recall Weight   : {recall_weighted:.4f}')
print(f'Stage 2 Test F1-score Weight : {f1_weighted:.4f}')

labels_in_test = sorted(y_test_sport.unique())
target_names_in_test = [sport_id_to_label[label] for label in labels_in_test]

print('\nDetailed classification report:')
print(
    classification_report(
        y_test_sport,
        y_test_sport_pred,
        labels=labels_in_test,
        target_names=target_names_in_test,
        zero_division=0
    )
)

cm = confusion_matrix(y_test_sport, y_test_sport_pred, labels=labels_in_test)
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=target_names_in_test,
    yticklabels=target_names_in_test
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Stage 2 Confusion Matrix (Test Set)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Per-class metrics, easier to inspect than the full text report
sport_report = classification_report(
    y_test_sport,
    y_test_sport_pred,
    labels=labels_in_test,
    target_names=target_names_in_test,
    zero_division=0,
    output_dict=True
)

sport_metrics = (
    pd.DataFrame(sport_report)
    .T
    .loc[target_names_in_test, ['precision', 'recall', 'f1-score', 'support']]
    .sort_values('f1-score')
)

display(sport_metrics.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
sport_metrics[['precision', 'recall', 'f1-score']].plot(kind='bar', ax=ax)
ax.set_title('Stage 2 Metrics by Sport Class')
ax.set_xlabel('Sport label')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## 7. End-to-End Pipeline Evaluation

In [ ]:
def predict_pipeline(texts):
    if VECTORIZER_TYPE == 'TF-IDF':
        binary_features = binary_vectorizer.transform(texts)
    else:
        binary_features = _docs_to_matrix_pretrained(texts, ft_model, W2V_VECTOR_SIZE)

    binary_preds = binary_mlp.predict(binary_features)

    final_preds = []
    for text, binary_pred in zip(texts, binary_preds):
        if binary_pred == 0:
            final_preds.append('non-sport')
        else:
            if SPORT_VECTORIZER_TYPE == 'TF-IDF':
                sport_features = sport_vectorizer.transform([text])
            else:  # fastText pretrained (300d)
                sport_features = _docs_to_matrix_pretrained([text], ft_model, W2V_VECTOR_SIZE)

            sport_pred = sport_mlp.predict(sport_features)[0]
            final_preds.append(sport_id_to_label[sport_pred])

    return final_preds, binary_preds


print("=============================================================")
print(f'[{SPORT_VECTORIZER_TYPE}]')
pipeline_test_preds, binary_test_preds = predict_pipeline(test_df['combined_text'])
pipeline_test_true = test_df['final_label'].tolist()

pipeline_acc = accuracy_score(pipeline_test_true, pipeline_test_preds)
pipeline_precision_macro, pipeline_recall_macro, pipeline_f1_macro, _ = precision_recall_fscore_support(
    pipeline_test_true,
    pipeline_test_preds,
    average='macro',
    zero_division=0
)
pipeline_precision_weighted, pipeline_recall_weighted, pipeline_f1_weighted, _ = precision_recall_fscore_support(
    pipeline_test_true,
    pipeline_test_preds,
    average='weighted',
    zero_division=0
)

print(f'Pipeline Test Accuracy        : {pipeline_acc:.4f}')
print(f'Pipeline Test Precision Macro : {pipeline_precision_macro:.4f}')
print(f'Pipeline Test Recall Macro    : {pipeline_recall_macro:.4f}')
print(f'Pipeline Test F1-score Macro  : {pipeline_f1_macro:.4f}')
print(f'Pipeline Test Precision Weight: {pipeline_precision_weighted:.4f}')
print(f'Pipeline Test Recall Weight   : {pipeline_recall_weighted:.4f}')
print(f'Pipeline Test F1-score Weight : {pipeline_f1_weighted:.4f}')

pipeline_labels = ['non-sport'] + sport_class_names
print('\nDetailed pipeline classification report:')
print(classification_report(pipeline_test_true, pipeline_test_preds, labels=pipeline_labels, zero_division=0))


## 8. Try New Examples

In [ ]:
new_samples = [
    'Manchester United won 3-1 with a late goal in the Premier League.',
    'The batter scored a century after a dominant opening partnership in the second innings.',
    'The driver secured pole position and managed tyre wear perfectly over the final laps.',
    'She broke serve twice to win the second set and reach the quarter-final.',
    'Donald Trump just died in his house in Florida.',
    'Bankrupt company announces restructuring plan for next quarter.'
]

pipeline_preds, stage1_preds = predict_pipeline(new_samples)

# Correctly generate features for predict_proba based on the active VECTORIZER_TYPE
if VECTORIZER_TYPE == 'TF-IDF':
    binary_features_for_proba = binary_vectorizer.transform(new_samples)
else:
    binary_features_for_proba = _docs_to_matrix_pretrained(new_samples, ft_model, W2V_VECTOR_SIZE)

stage1_probs = binary_mlp.predict_proba(binary_features_for_proba)[:, 1]

for text, stage1_pred, stage1_prob, final_pred in zip(new_samples, stage1_preds, stage1_probs, pipeline_preds):
    stage1_label = 'sport' if stage1_pred == 1 else 'non-sport'
    print(f'Text: {text}')
    print(f'Stage 1 prediction: {stage1_label} (sport probability={stage1_prob:.3f})')
    print(f'Final pipeline label: {final_pred}')
    print('-' * 80)


# 8. BERT


In [ ]:
!pip install -q transformers torch scikit-learn


In [ ]:

!pip install -q transformers datasets

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Config
MODEL_NAME   = "bert-base-uncased"
MAX_LEN      = 256        # titolo + testo tende ad essere lungo
BATCH_SIZE   = 16
EPOCHS       = 4
LR           = 2e-5
WARMUP_RATIO = 0.1
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES  = len(sport_class_names)

print(f"Device: {DEVICE} | Classi sport: {NUM_CLASSES} → {sport_class_names}")

# Dataset
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

class SportDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }

train_ds = SportDataset(X_train_sport, y_train_sport, tokenizer, MAX_LEN)
val_ds   = SportDataset(X_val_sport,   y_val_sport,   tokenizer, MAX_LEN)
test_ds  = SportDataset(X_test_sport,  y_test_sport,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Modello
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label=sport_id_to_label,
    label2id=sport_label_to_id,
)
model.to(DEVICE)

# Optimizer + Scheduler
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# Training loop
def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, leave=False, disable=not training):
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            lbls  = batch["labels"].to(DEVICE)

            out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
            loss = out.loss

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item() * ids.size(0)
            preds       = out.logits.argmax(dim=-1)
            correct    += (preds == lbls).sum().item()
            total      += ids.size(0)

    return total_loss / total, correct / total


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    vl_loss, vl_acc = run_epoch(val_loader,   training=False)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)

    print(f"Epoch {epoch}/{EPOCHS}  "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
          f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.4f}")

# Evaluation su test set
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test", leave=False):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)

        logits = model(input_ids=ids, attention_mask=mask).logits
        preds  = logits.argmax(dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())

model.save_pretrained("./bert_sport_stage2")
tokenizer.save_pretrained("./bert_sport_stage2")
print("Modello salvato in ./bert_sport_stage2")

In [ ]:
bert_history = pd.DataFrame(history)
bert_history.insert(0, 'epoch', np.arange(1, len(bert_history) + 1))
display(bert_history.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bert_history['epoch'], bert_history['train_acc'], marker='o', label='Train accuracy')
ax.plot(bert_history['epoch'], bert_history['val_acc'], marker='o', label='Validation accuracy')
ax.set_title('BERT Stage 2 - Accuracy by Epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_xticks(bert_history['epoch'])
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bert_history['epoch'], bert_history['train_loss'], marker='o', label='Train loss')
ax.plot(bert_history['epoch'], bert_history['val_loss'], marker='o', label='Validation loss')
ax.set_title('BERT Stage 2 - Loss by Epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_xticks(bert_history['epoch'])
ax.legend()
plt.tight_layout()
plt.show()


Nota: il modello pretrained è stato salvato su drive. Lo si può recuperare

In [ ]:
# Classification Report
unique_ids   = sorted(set(all_labels))
unique_names = [sport_id_to_label[i] for i in unique_ids]

print("\n── Classification Report ──")
print(classification_report(
    all_labels, all_preds,
    labels=unique_ids,          # solo le classi presenti nel test
    target_names=unique_names,
    zero_division=0,
))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds, labels=unique_ids)
fig, ax = plt.subplots(figsize=(max(6, len(unique_ids)), max(5, len(unique_ids) - 1)))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=unique_names,
    yticklabels=unique_names,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Stage 2 – Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Model & Strategy Comparison

This section compares different models and strategies using key metrics such as accuracy, precision, recall, and F1-score.


In [ ]:

import matplotlib.pyplot as plt

# Example: replace these with actual results from your notebook
models = ['Baseline', 'TF-IDF + LR', 'TF-IDF + SVM', 'Two-Stage Model']
accuracy = [0.75, 0.88, 0.90, 0.92]
f1_scores = [0.74, 0.87, 0.89, 0.91]

# Accuracy comparison
plt.figure()
plt.bar(models, accuracy)
plt.title("Model Accuracy Comparison")
plt.xlabel("Models")
plt.ylabel("Accuracy")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# F1-score comparison
plt.figure()
plt.bar(models, f1_scores)
plt.title("Model F1 Score Comparison")
plt.xlabel("Models")
plt.ylabel("F1 Score")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()



## Dummy Baseline

Questa baseline usa un modello volutamente semplice: `DummyClassifier`.

Serve come riferimento minimo: un modello reale dovrebbe superare chiaramente questa baseline.


In [ ]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Cerca automaticamente nomi comuni già presenti nel notebook.
# Se nel tuo notebook usi nomi diversi, modifica solo queste quattro variabili.
try:
    X_train_baseline = X_train
    X_test_baseline = X_test
    y_train_baseline = y_train
    y_test_baseline = y_test
except NameError:
    try:
        X_train_baseline = X_train_tfidf
        X_test_baseline = X_test_tfidf
        y_train_baseline = y_train
        y_test_baseline = y_test
    except NameError:
        raise NameError(
            "Non trovo X_train/X_test/y_train/y_test. "
            "Aggiorna i nomi delle variabili nella cella della Dummy Baseline."
        )

dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train_baseline, y_train_baseline)

y_pred_dummy = dummy.predict(X_test_baseline)

dummy_accuracy = accuracy_score(y_test_baseline, y_pred_dummy)
dummy_f1_macro = f1_score(y_test_baseline, y_pred_dummy, average="macro")
dummy_f1_weighted = f1_score(y_test_baseline, y_pred_dummy, average="weighted")

print("Dummy baseline — strategy: most_frequent")
print(f"Accuracy:    {dummy_accuracy:.4f}")
print(f"F1 macro:    {dummy_f1_macro:.4f}")
print(f"F1 weighted: {dummy_f1_weighted:.4f}")

print("\nClassification report:")
print(classification_report(y_test_baseline, y_pred_dummy, zero_division=0))


In [ ]:

ConfusionMatrixDisplay.from_predictions(
    y_test_baseline,
    y_pred_dummy,
    xticks_rotation=45,
    values_format="d"
)
plt.title("Dummy Baseline — Confusion Matrix")
plt.tight_layout()
plt.show()



## Updated Comparison Including Dummy Baseline

Il grafico seguente include la dummy baseline come riferimento minimo.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

baseline_results = pd.DataFrame([
    {
        "model": "Dummy most_frequent",
        "accuracy": dummy_accuracy,
        "f1_macro": dummy_f1_macro,
        "f1_weighted": dummy_f1_weighted,
    }
])

display(baseline_results)

# Grafico pulito delle metriche principali della baseline
metrics = ["accuracy", "f1_macro", "f1_weighted"]
plt.figure()
plt.bar(metrics, baseline_results.loc[0, metrics])
plt.title("Dummy Baseline Metrics")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# Se esiste già un DataFrame di risultati nel notebook, prova ad aggiungere la baseline.
# Altrimenti crea una base riutilizzabile.
candidate_names = ["results_df", "comparison_df", "model_results_df", "results"]
existing_df = None

for name in candidate_names:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        existing_df = globals()[name].copy()
        break

if existing_df is not None:
    combined_results = pd.concat([baseline_results, existing_df], ignore_index=True)
else:
    combined_results = baseline_results.copy()

display(combined_results)

available_metrics = [m for m in ["accuracy", "f1_macro", "f1_weighted", "f1_score", "f1"] if m in combined_results.columns]

for metric in available_metrics:
    plt.figure()
    plt.bar(combined_results["model"], combined_results[metric])
    plt.title(f"Model Comparison — {metric}")
    plt.xlabel("Model")
    plt.ylabel(metric)
    plt.ylim(0, 1)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()
